# 11. Conclusions: the models we keep

A summary of the data analysis (nb 01–09): **every model we keep, every parameter, its value, where it came from, and how far to trust it.** Nothing is fitted here. Numbers come from the notebooks named next to them; the tables at the end of some sections read the data files those notebooks save, so they stay in sync.

1. The energy balance in one line
2. Load (nb 06, nb 07)
3. Solar (nb 09): the pseudo-forecast
4. Battery (nb 10)
5. Comms: the Starlink schedule (nb 04)
6. The mission (nb 05)
7. Currents (nb 08)
8. How good it is so far (nb 12)
9. What we assume, and what's still open

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 1. The energy balance in one line

Every minute:

**P_net = η · P_solar − P_load − P_drain**  (W into the battery)

**OCV += P_net · Δt / k**, and the terminal voltage is **V = OCV + R · I**, with I = P_net / V. No charging above **25.0 V**.

- **P_load** = 10 W hotel + 22 W × Starlink on + k_mode × v_line (section 2)
- **P_solar** = the solar **pseudo-forecast**: one smooth curve per day (section 3)
- **η, P_drain, k, R** = the battery model (section 4)

The inputs that drive it over time: **mode and arming** (motor on or off), **speed**, **Starlink on/off**, and **how sunny the day is**. The mission (section 6) and the Starlink schedule (section 5) give the first three.

## 2. Load (nb 06, nb 07)

**P_load = P_hotel + P_starlink · Starlink + k_mode · v_line**

| parameter | value | where it comes from | how sure |
| --- | --- | --- | --- |
| **P_hotel** | **10 W** | MANUAL with Starlink off: load is flat with speed there, so the motor is off and what's left is the base load (computer, autopilot, radios) (nb 06) | moves between ~5 and 15 W over a day |
| **P_starlink** | **22 W** | MANUAL, Starlink on minus Starlink off, the same at every speed (nb 06). Iridium-only idle ≈ 11 W vs ≈ 33 W on Starlink | anything that switches on with Starlink is counted in here |
| **k_AUTO** | **71.3 ± 0.7 W per m/s** | least squares of propulsion = load − 10 − 22·Starlink against the **mission line speed**, AUTO minutes only, **no intercept** (hotel and Starlink locked) (nb 07 section 4) | 29 lines, R² 0.71 over the line means |
| **k_mode17** | **54.4 ± 1.1 W per m/s** | same, mode 17 minutes only (nb 07 section 4) | 10 lines, R² 0.85 over the line means |
| **k_MANUAL** | **0** | MANUAL armed or disarmed draws ~0 W of propulsion at any speed (nb 06 section 2) | |
| **v_line** | m/s | the speed of the mission line the minute is in: median speed over ground of that line (nb 05, `mission_modes.csv`) | includes current and wind (section 7) |
| **Starlink** | 0 or 1 | 1 inside a Starlink session (messages < 2 min apart, ± 30 s); in the forecast, the schedule (section 5) | |

**Why the line speed and not the per-minute speed (nb 07):** averaging over a line removes the speed noise (waves, turns, GPS). The line dots follow k·v with R² 0.76 (one k for both modes), against 0.58 for the minutes in nb 06. It's also the only speed the forecast has. Predicting the per-minute load with real Starlink: RMSE 19.0 W with the line speed and k per mode, vs 18.0 W with the real per-minute speed.

**Why k·v:** it beats k·v² and k·v³ on the per-minute speed (nb 06: R² 0.58 vs 0.41 and 0.05) and on the line speed, in both modes (nb 07).

**Why one k per mode:** at the same speed mode 17 draws less than AUTO (about 17 W less at 1 m/s). One shared k (66.8) leaves AUTO 3 W too low and mode 17 10 W too high; per mode both are within 2 W.

**Known limits:**
- **Holding and drifting counts as the motor running.** Lines where the boat holds in mode 17 or AUTO move at 0.3 m/s with the current while the motor draws 2–5 W, and k·v gives them ~16–18 W (16–17 Jul, nb 12 section 1).
- **The real curve looks like a step**: ~5 W below 0.5 m/s, a flat top ~105 W from 1.2 m/s (nb 06).
- **The Starlink schedule misses the extra sessions** switched on from shore: with the schedule instead of real Starlink, the load RMSE goes from 19.0 to 22.4 W (nb 07). This is now the largest load error.
- The old per-minute fit (nb 06, k·v 64.8 W per m/s) is kept as a reference.

## 3. Solar (nb 09): the pseudo-forecast

**P_solar(t) = peak_day · cos(π · (t − t_noon) / L)ᵖ** inside the day, **0 at night** (t = UTC hour).

| parameter | value | where it comes from | how sure |
| --- | --- | --- | --- |
| **t_noon** | **12.7 h UTC** | grid search, shared by all days, on per-minute means of battery3 V × A (Starlink rows only) | solar noon at 12:42 UTC hints the real position is a few degrees west of Greenwich (positions are shifted) |
| **L** | **24 h** | same grid search (top of the grid; with p = 5.5 the curve is a bell and already ~0 at night) | the curve is above 2 % of its peak from 04:36 to 20:48 UTC, matching the solar day in the data |
| **p** | **5.5** | same grid search | |
| **peak_day** | **52–172 W**, one per day (table below) | least squares on one number per day, for the shared shape. Days with < 30 daylight minutes get the **mean, 100 W** | uses the day's own data: a **fit**, not a forecast |

Per-minute error over all Starlink minutes: **20.5 W RMSE** with one curve per day, against **33.8 W** for the old fixed curve (same every day) and **32.5 W** for the per-day shape with **yesterday's** peak (the simplest real forecast).

**Assumption for the forecast (our call):** positions and dates are shifted, so no weather forecast matches the real days. We **treat the fitted daily peaks as the weather forecast, and assume it came true.** The forecast's error then measures everything *except* the weather. This must be said plainly in the write-up.

**Also kept:**
- **Irradiance → power** (for a real weather forecast on the boat, e.g. Open-Meteo): **P = k · G**, with **k = η·A = 0.164 m²** (measured Wh/day ÷ ERA5 July Wh/m²/day at the nominal position; the peak-hour method gives 0.210 m²). This is what the planner and the simulator use.

In [2]:
with open("../data/solar_daily_peaks.csv") as f:
    print(f.readline().strip())
peaks = pd.read_csv("../data/solar_daily_peaks.csv", comment="#", index_col="day")
peaks.T

# shape = cos(pi*(hour_utc - t_noon)/L)^p inside the day, 0 at night; t_noon=12.70 L=24.00 p=5.50 mean_peak_W=100.1


day,2025-07-14,2025-07-15,2025-07-16,2025-07-17,2025-07-18,2025-07-19,2025-07-20,2025-07-21,2025-07-22,2025-07-23,2025-07-24,2025-07-25,2025-07-26,2025-07-27
peak_W,142.8,172.4,110.1,58.5,90.4,137.7,98.4,80.0,119.3,72.9,52.3,63.7,102.4,NaN


## 4. Battery (nb 10), model B1

| parameter | value | where it comes from | how sure |
| --- | --- | --- | --- |
| **k** | **420 ± 31 Wh per V** | change in open-circuit voltage vs net energy over 22 windows (14 night, 8 day), OLS | the main number. Implies ~1–1.5 kWh usable between 20 and 25 V |
| **η** | **1.14 ± 0.07** | same fit: solar → battery factor | **> 1 is not physical.** It soaks up a scale error in the solar or load current sensors. Keep it as a calibration number, not an efficiency |
| **P_drain** | **−0.3 ± 2.1 W** | same fit: a constant drain not seen in the load reading | zero within its error |
| **R** | **39.2 mΩ**, fixed | within-window regression of voltage on current (r 0.95) | fixed before the fit, not fitted with k (when fitted, R doubled and the free run got worse) |
| **V_cap** | **25.0 V** | the battery stops rising there (28–30 Jul plateau ~25.2 V terminal) | |
| **battery_pct** | **clip(20 · V − 400, 0, 100)** | exact, from the telemetry | it's the voltage, not a real state of charge |

**Why B1:** of the 4 variants (R fitted or fixed, OLS or Huber), B1 gives the best free run with **measured** solar and load: **RMSE 0.40 V** over 14 Jul to 27 Jul.

**Known limit:** fitted on short windows; over many days the inputs disagree with the voltage (nb 12 section 3), so k or η may be slightly off for long runs.

## 5. Comms: the Starlink schedule (nb 04)

Starlink is the second-biggest load after the motor (22 W while on). It runs on a schedule that the boat sets, not the mission:

- **Deployment, 14 Jul 07:43 to 15 Jul 20:00:** on for hours, no schedule → saved as a **duty cycle** (share of time on).
- **15 Jul 20:00 to 19 Jul 22:00:** every **2 h** at even UTC hours, wakes at **hh:47**, ~8 min.
- **19 Jul 22:00 to 27 Jul 02:05:** every **3 h** at 01, 04, …, 22 h, wakes at **hh:47**, ~16 min.
- **After 27 Jul 02:05:** never again (off).

On top: 60 of 159 sessions are **extra**, off the schedule (mostly daytime, probably switched on from shore). They are not in the schedule.

Iridium (always on, ~every 2–15 min) and Globalstar (~every 4 h) are inside the 10 W hotel load.

In [3]:
pd.read_csv("../data/mission_starlink.csv")

,from,to,wake_hours,on_min,duty
0,2025-07-14 00:00:00+00:00,2025-07-15 20:00:00.000000+00:00,NaN,0,0.435864
1,2025-07-15 20:00:00+00:00,2025-07-19 22:00:00.000000+00:00,00 02 04 06 08 10 12 14 16 18 20 22,8,0.000000
2,2025-07-19 22:00:00+00:00,2025-07-27 02:05:54.366297+00:00,01 04 07 10 13 16 19 22,16,0.000000
3,2025-07-27 02:30:00+00:00,2025-07-31 00:00:00.000000+00:00,NaN,0,0.000000


## 6. The mission (nb 05)

What the boat was told to do, rebuilt from the telemetry: **one line per change of mode or arming**, nothing smoothed. **Every change is treated as the operator's command** (on purpose), even blips of a few seconds. The forecast predicts energy *given* what the boat was told to do.

- **95 lines**, 14 Jul 00:20 to 30 Jul 23:42.
- Each line: from, to, mode, armed, and (for checking only) what happened: median speed over ground and motor power.
- **For the load model:** MANUAL (armed or disarmed) = motor off; AUTO and mode 17 = motor on at the line's speed, with k_AUTO 71.3 or k_mode17 54.4 W per m/s (section 2).
- **We only see a change at the next message:** 1–15 s over Starlink, up to 15 min over Iridium alone (13 changes may be > 5 min late).

**Short MANUAL blips** (27 lines under 15 min, ~1.5 h in total) come back to the same mode, all in daytime, often with a waypoint change: most likely operator actions, not faults. Too short to matter for energy.

In [4]:
mission = pd.read_csv("../data/mission_modes.csv", parse_dates=["from", "to"])
mission["category"] = mission["mode"].where(mission["mode"] != "MANUAL", "MANUAL " + mission.armed.map({True: "armed", False: "disarmed"}))
summary = mission.groupby("category").agg(lines=("hours", "size"), hours=("hours", "sum"),
                                         median_SOG=("SOG (m/s)", "median"), median_motor_W=("motor (W)", "median"))
summary.round(1)

,lines,hours,median_SOG,median_motor_W
category,,,,
AUTO,34,97.3,0.9,43.7
INITIALISING,2,0.1,0.0,NaN
MANUAL armed,35,167.6,0.6,4.8
MANUAL disarmed,13,44.5,0.5,1.2
mode 17,11,97.8,0.9,40.0


## 7. Currents (nb 08)

Not in the forecast yet, but they explain part of the load scatter.

- **Drift with the motor off** (MANUAL legs ≥ 30 min, 14 Jul tow left out): time-weighted mean **0.20 m/s towards 96°** (east). Armed (0.22 m/s → 95°) and disarmed (0.13 m/s → 109°) agree.
- **Hour by hour** the east part stays mostly positive (+0.1 to +0.6 m/s) while the north part swings ±0.3–0.6 m/s: a tide-like turning on top of a steady eastward set.
- **Power per m/s over ground by heading** (motor on): cheap towards east–south (18–36 W per m/s), expensive towards west–north (59–82). The cheap side lies around the drift direction: two independent views agree.

**What that means for the load model:** k·v uses speed over ground, so the same motor power gives more speed with the current and less against it. That's part of the 24 W per-minute scatter. A speed-through-water model would need the current at every minute, which we only have on average.

**Caveat:** heading is mixed up with date (northbound mostly in the first week); the east–south bins are small (55–70 min).

## 8. How good it is so far (nb 12)

nb 12 runs the forecast with the **mission load** (section 2: mission lines, k per mode, Starlink schedule), the **pseudo-forecast solar** (section 3) and battery B1 (section 4).

**The forecast:** 24 h pieces from 00:00 each day, starting from the measured voltage: **mean |end error| 0.39 V, mean end error −0.01 V, end error RMSE 0.53 V**. One 17-day free run, never corrected: **RMSE 1.26 V**.

**Isolation** (to 27 Jul, while solar is measured):

| run | solar | load | 24 h mean \|end error\| | 17 days RMSE |
| --- | --- | --- | --- | --- |
| **forecast** | pseudo-forecast | mission | 0.40 V | 1.26 V |
| perfect solar | measured | mission | 0.28 V | 1.01 V |
| perfect load | pseudo-forecast | measured | 0.23 V | 0.66 V |
| both measured | measured | measured | 0.16 V | 0.40 V |

**Reading:**
- **The pseudo-forecast solar matches the measured solar energy** (11.55 vs 11.54 kWh to 27 Jul).
- **The mission load has no bias** (mean end error −0.01 V) and the total load is close (13.0 vs 12.7 kWh to 27 Jul). The day-to-day error comes from holding/drifting lines counted as motor on (17 Jul +1.5 V) and from Starlink sessions the schedule misses.
- **The battery model is fine:** with both inputs measured, 0.16 V per day.

**Caveats:** the daily solar peaks, the mission speeds and the motor curves all come from the same 17 days we forecast. This is the error *if the plan and the weather forecast were right*. There's still no error on data the models weren't fitted to.

## 9. What we assume, and what's still open

**Assumptions (our calls, to state in the write-up):**
- Every mode or arming change in the telemetry was a command (section 6).
- The fitted daily solar peaks are the weather forecast, and it came true (section 3).
- MANUAL armed and disarmed = motor off (section 2).
- The mission line's speed is the speed the mission asked for (section 2).
- Hotel 10 W and Starlink 22 W are locked from the MANUAL tests; no fit may move them.

**Open hypotheses (to test):**
- **Below ~0.4 m/s the speed is drift, not the motor.** *Test:* motor = 0 for lines under 0.4 m/s, or k · (v − v_drift) with v_drift ≈ 0.2–0.3 m/s; refit k per mode (nb 07) and rerun nb 12.
- **Held-out error:** fit on 14–22 Jul, test on 23–30 Jul. *Test:* refit load, solar shape and battery on the first half; forecast the second half; compare with the all-days numbers.
- **The north–south swing in the drift is the tide.** *Test:* fit a 12.42 h sine to the hourly north drift of the longest legs; check that its phase is the same across legs.
- **Short MANUAL blips are operator actions.** *Test:* blips with a waypoint change should show the new waypoint right after; motor-on blips should show turns AUTO wouldn't make.
- **η > 1 is a sensor scale error.** *Test:* the energy balance P_batt ≈ 0.98·P_solar − P_load (nb 01–03) already suggests the currents are consistent; check whether η drops to ~1 when R and k are fitted on long windows only.